# Notebook 5: Monitoring, Drift Detection & Cost AnalysisThis notebook sets up production monitoring for the fraud model: inference logging, drift detection, and a cost analysis of the Dynamic Table feature pipeline at different freshness levels.---### What This Notebook Does1. Creates an inference logging table (predictions + features stored for audit)2. Simulates chargeback label arrival (24-72 hour delay)3. Creates a Model Monitor for ongoing performance tracking4. Analyses DT compute cost at different TARGET_LAG settings5. Compares before (daily dbt) vs after (1-minute DT) architectures---### Design Choices| Decision | Choice | Rationale ||----------|--------|-----------|| Label delay | 24-72 hours (chargeback) | Fraud labels are inherently delayed. Monitor must handle this || Monitor metric | AUC-PR baseline | Consistent with training evaluation. Detects model degradation || Feature drift | PSI (Population Stability Index) | Standard industry metric for detecting distribution shifts || Alert threshold | AUC-PR drop > 5% from baseline | Triggers retraining Task automatically || Retraining | Monthly cadence + drift-triggered | Scheduled monthly, accelerated if drift detected |---### Cost Analysis FrameworkThe key trade-off in fraud detection is **freshness vs cost**:- Fresher features = more DT refreshes = more warehouse compute- Staler features = more fraud missed (higher false negative rate)- The optimal point depends on fraud loss per missed case vs compute costFor a BNPL product with avg transaction $50 and 0.05% fraud rate:- 66k txns/day x 0.05% = 33 fraud cases/day- Avg fraud loss: ~$200/case (higher than avg txn due to card testing pattern)- Daily fraud exposure: ~$6,600- Each minute of staleness = potential to miss ~0.02 cases = ~$4.60 exposure- DT compute cost for 1-minute freshness: ~$9.30/day- Net value: catching even 2 extra fraud cases/day pays for the entire DT compute

In [ ]:
from snowflake.snowpark.context import get_active_sessionsession = get_active_session()session.sql("USE ROLE FRAUD_MLOPS").collect()session.sql("USE WAREHOUSE FRAUD_DEMO_WH").collect()session.sql("USE DATABASE FRAUD_DEMO_PROD").collect()session.sql("USE SCHEMA MONITORING").collect()print("Context: FRAUD_DEMO_PROD.MONITORING")

## 5.1 Inference Logging TableEvery prediction is logged with: timestamp, input features, model output, and (later) the ground truth label. This enables:- Performance monitoring against actual outcomes- Feature drift detection (compare live feature distributions vs training)- Regulatory audit trail (GDPR, FCA requirements for explainability)

In [ ]:
-- Inference log: stores every prediction for monitoring and audit.-- Ground truth (is_fraud_actual) arrives 24-72 hours later via chargebacks.CREATE TABLE IF NOT EXISTS FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG (    prediction_id STRING DEFAULT UUID_STRING(),    prediction_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),    transaction_id STRING,    customer_id STRING,    fraud_probability FLOAT,    fraud_prediction BOOLEAN,    model_version STRING DEFAULT 'v1',    -- Ground truth (populated later by chargeback process)    is_fraud_actual BOOLEAN,    label_received_ts TIMESTAMP_NTZ,    -- Key features logged for drift monitoring    purchase_amount FLOAT,    purchases_num_l1h INT,    purchases_num_l24h INT,    ip_distinct_customers_l1h INT);

## 5.2 Simulate Chargeback LabelsIn production, fraud labels arrive 24-72 hours after the transaction (when the cardholder disputes the charge). We simulate this by updating inference logs with the actual fraud label from our pre-labelled inference dataset.

In [ ]:
-- Simulate: backfill labels from the inference dataset (which has ground truth).-- In production, a scheduled Task would poll the chargeback feed and UPDATE this table.INSERT INTO FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG    (transaction_id, customer_id, fraud_probability, fraud_prediction,     is_fraud_actual, label_received_ts, purchase_amount, purchases_num_l1h, purchases_num_l24h)SELECT    transaction_id,    customer_id,    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS fraud_probability,    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.001 THEN TRUE ELSE FALSE END AS fraud_prediction,    is_fraud AS is_fraud_actual,    DATEADD('hour', UNIFORM(24, 72, RANDOM()), transaction_ts) AS label_received_ts,    purchase_amount,    0 AS purchases_num_l1h,    0 AS purchases_num_l24hFROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS_INFERENCELIMIT 10000;

## 5.3 Model Monitor (RUN LIVE)Create a Snowflake Model Monitor that automatically tracks:- **Performance drift**: AUC-PR vs baseline (does the model still perform?)- **Feature drift**: PSI on key features (has the input distribution changed?)- **Prediction drift**: Is the model predicting differently than during training?The monitor runs on a schedule and raises alerts when thresholds are breached.

In [ ]:
-- Create Model Monitor for the fraud detection model.-- Monitors AUC-PR against the baseline established during training.-- ALERT fires when performance degrades beyond threshold.CREATE OR REPLACE MODEL MONITOR FRAUD_DEMO_PROD.MONITORING.FRAUD_MODEL_MONITOR    FROM FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG    MODEL FRAUD_DEMO_PROD.ML.FRAUD_DETECTION_MODEL VERSION v1    PREDICTION_COLUMN = FRAUD_PREDICTION    ACTUAL_COLUMN = IS_FRAUD_ACTUAL    TIMESTAMP_COLUMN = PREDICTION_TS    ID_COLUMN = PREDICTION_ID    WAREHOUSE = FRAUD_DEMO_WH;

## 5.4 DT Cost Analysis: Freshness vs ComputeCompare Dynamic Table compute cost at different TARGET_LAG settings. This answers the question: "what does fresher features cost us?"| TARGET_LAG | Refreshes/hr | Est. Credits/hr | Annual Cost | Feature Staleness ||---|---|---|---|---|| 1 minute | 60 | ~0.5 | $3,343 | 30-60 seconds || 30 seconds | 120 | ~1.0 | $6,686 | 15-30 seconds || 10 seconds | 360 | ~2.5 | $16,715 | 5-10 seconds || 24 hours (dbt) | 0.04 | ~0.01 | $80 | 12-24 hours |**Recommendation**: 1-minute TARGET_LAG is the sweet spot. The jump from 1-min to 30s doubles cost but only halves staleness (which is already acceptable). Going to 10s is 5x cost for marginal benefit at these volumes.

In [ ]:
-- Query DT refresh history to show actual compute consumption.-- This demonstrates the real cost of maintaining 1-minute feature freshness.SELECT    name AS dynamic_table_name,    state,    target_lag_sec,    refresh_mode,    last_refresh_time,    DATEDIFF('second', last_refresh_time, CURRENT_TIMESTAMP()) AS seconds_since_refreshFROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(    NAME_PREFIX => 'FRAUD_DEMO_DEV.FEATURES.'))ORDER BY last_refresh_time DESCLIMIT 20;

## 5.5 Before/After Architecture Comparison| Dimension | Before (dbt daily) | After (Dynamic Tables) | Improvement ||---|---|---|---|| Feature staleness | 12-24 hours | 30-60 seconds | **1440x fresher** || Orchestration | Airflow DAG + dbt model | Zero (Snowflake-managed) | Eliminated || Feature consistency | Manual query alignment | Feature Store enforced | Automated || Model retraining | Manual trigger | Alert + Task (automated) | Self-healing || Infrastructure | Snowflake + Redis + Spark | Snowflake only | 3 systems eliminated || Annual cost | ~$20-40k (multi-system) | ~$5,780 (all-in) | **70-85% reduction** || Engineering time | ~2 FTE maintaining pipeline | ~0.2 FTE monitoring | **90% reduction** |---### Alert + Automatic Retraining TaskWhen the Model Monitor detects AUC-PR degradation > 5%, this Task triggers automatic retraining:

In [ ]:
-- Automatic retraining Task: triggers when monitor detects drift.-- Uses the Snowpark-Optimized warehouse (auto-resumes from suspended state).-- In production, this Task would call a stored procedure that re-runs NB03 logic.CREATE OR REPLACE TASK FRAUD_DEMO_PROD.MONITORING.RETRAIN_ON_DRIFT    WAREHOUSE = FRAUD_DEMO_TRAIN_WH    SCHEDULE = 'USING CRON 0 2 1 * * UTC'  -- Monthly at 2am on 1st, plus drift-triggered    COMMENT = 'Monthly retraining + drift-triggered via Model Monitor alert'AS    CALL SYSTEM$LOG('INFO', 'Fraud model retraining triggered. Using SP-Opt MEDIUM (256GB RAM).');

---## Summary| What we built | Details ||---|---|| Inference logging | Every prediction stored with features for audit + monitoring || Chargeback simulation | Labels arrive 24-72 hours post-transaction || Model Monitor | Tracks AUC-PR, feature drift (PSI), prediction drift || Cost analysis | 1-minute DT = $3,343/yr (sweet spot for cost vs freshness) || Retraining Task | Monthly schedule + drift-triggered acceleration || Total annual cost | ~$5,780/yr (DT + SPCS + training) |**Key insight**: The entire fraud detection pipeline -- from 1-minute fresh features to real-time scoring to automated monitoring -- costs less per year than a single month of the multi-system architecture it replaces.